# Qwen2.5-0.5B on recent movies: train, evaluate, export (Google Colab)

One notebook, top to bottom: fine-tune Qwen2.5-0.5B on the recent-movies dataset (MovieLens 32M cut to 1M ratings,
movies 2010-2023 - `01_prep.ipynb`, `02_tmdb_metadata.ipynb`), test it on the same 1,000 users as SASRec and TFRS,
and **write `qwen.csv` automatically** (each user's top 10 + probabilities, for the API) to Drive and to your
computer.

| task | question | answer |
|---|---|---|
| `plot` | What is the movie "*title*" about? | TMDB overview |
| `genres` | Which genres does "*title*" belong to? | genres |
| `details` | Who made "*title*" and who stars in it? | director + main cast |
| `identify` | Which movie is this? *plot snippet* | title (year) |
| `next_movie` | A user rated these movies, oldest first ... which next? | title (year) |

**Before you run**
1. Upload the folder `genrec/data/ml-32m-recent/` from your Mac (`ratings.dat` + `movies_wiki.csv`) into your Drive
   project folder, so it is `<DRIVE_DIR>/ml-32m-recent/`.
2. Runtime → Change runtime type → GPU (**A100** or **L4**).
3. Runtime → Run all. If it stops with "transformers was already imported": Runtime → Restart session, Run all.

**Time:** training ~2-3 h on an L4 (about half on an A100), then ~30 min to evaluate and score. Keep the tab open.
The trained model is saved to Drive: if the runtime disconnects after training, Run all again - it skips training
when the model is already there and goes straight to evaluation and the CSV.

In [ ]:
import sys

already_loaded = "transformers" in sys.modules   # an old version in memory would mix with the new files
GENREC_REPO = "https://github.com/mazprimrez/project-playground"

!nvidia-smi --query-gpu=name,memory.total --format=csv
# clean reinstall of the versions the code was tested with (Colab ships its own), then the genrec package
%pip install -q --force-reinstall --no-deps transformers==5.18.0 accelerate==1.15.0 peft==0.21.1
%pip install -q "genrec[train] @ git+{GENREC_REPO}#subdirectory=genrec" tqdm

assert not already_loaded, "transformers was already imported: Runtime -> Restart session, then Run all again"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
# your project folder on Drive (Files panel -> right-click the folder -> Copy path)
DRIVE_DIR = Path("/content/drive/MyDrive/Self-learning AI related/GenRec - Netflix")
DATA_DIR = DRIVE_DIR / "ml-32m-recent"                         # ratings.dat + movies_wiki.csv
FINAL_DIR = DRIVE_DIR / "qwen2.5-0.5b-ml32m-recent" / "final"   # the trained model
OUT_CSV = DRIVE_DIR / "qwen.csv"                                 # top 10 + probabilities for the API
CHECKPOINT_DIR = Path("/content/checkpoints/genrec-ml32m")      # Colab disk: a full checkpoint is ~6 GB

for p in [DATA_DIR / "movies_wiki.csv", DATA_DIR / "ratings.dat"]:
    assert p.exists(), f"missing {p} - upload genrec/data/ml-32m-recent/ to your Drive folder"
RUN_TRAINING = not (FINAL_DIR / "config.json").exists()       # already trained: evaluate and export only
print("train, then evaluate" if RUN_TRAINING else f"found a trained model in {FINAL_DIR}: evaluate only")

## 1. Config

One stage from the base model. On ML-1M, more recommendation examples per user helped most (5 -> 20 took HR@10
from 0.160 to 0.236); 10 per user keeps this run at ~2-3 h on an L4. On an A100, `next_per_user=20` fits in a
similar time.

In [ ]:
import time

import pandas as pd
import torch
from transformers import AutoTokenizer, TrainingArguments, set_seed

from genrec.prompts import build_samples
from genrec.training import (AnswerOnlyTrainer, ChatDataset, PadCollator, TrainConfig, fact_recall, load_model,
                             show_generations)

assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
cfg = TrainConfig(
    model="Qwen/Qwen2.5-0.5B", data_dir=DATA_DIR, output_dir=CHECKPOINT_DIR,
    next_per_user=10,     # recommendation samples per user
    epochs=2,             # validation loss rose after 2 epochs on ML-1M
    lr=5e-5,              # from the base model
    batch_size=4, grad_accum=8,
)
set_seed(cfg.seed)
cfg

## 2. Train

Samples → model → checks (the answer-only loss matches the model's own loss; the longest batch fits in memory) →
train. The checkpoint with the lowest validation loss on 500 held-out users is kept. Re-running this cell after an
interruption resumes from the last checkpoint (if the runtime wasn't reset).

In [ ]:
if RUN_TRAINING:
    train_samples, eval_samples = build_samples(cfg)
    print("train:", pd.Series([s["task"] for s in train_samples]).value_counts().to_dict())
    print("eval: ", {k: len(v) for k, v in eval_samples.items()})

    tokenizer = AutoTokenizer.from_pretrained(cfg.model)
    train_ds = ChatDataset(train_samples, tokenizer, cfg.max_length)
    eval_ds = {k: ChatDataset(v, tokenizer, cfg.max_length) for k, v in eval_samples.items()}
    train_ds.describe("train")

    # T4 (compute capability 7.5): fp16 mixed precision; L4 / A100 (8.0+): bf16
    precision = "bf16" if torch.cuda.get_device_capability()[0] >= 8 else "fp16"
    model = load_model(cfg).to("cuda")
    collator = PadCollator(tokenizer.pad_token_id)
    best_metric = "eval_next_movie_loss" if "next_movie" in eval_ds else None
    training_args = TrainingArguments(
        output_dir=str(cfg.output_dir), num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.batch_size, per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum, learning_rate=cfg.lr, lr_scheduler_type="cosine",
        warmup_steps=cfg.warmup_ratio,   # transformers v5: a float in [0, 1) is a ratio of total steps
        weight_decay=cfg.weight_decay, logging_steps=cfg.logging_steps,
        eval_strategy="steps" if eval_ds else "no", eval_steps=cfg.eval_steps,
        save_strategy="steps", save_steps=cfg.save_steps, save_total_limit=2,
        load_best_model_at_end=best_metric is not None, metric_for_best_model=best_metric,
        fp16=precision == "fp16", bf16=precision == "bf16", gradient_checkpointing=cfg.gradient_checkpointing,
        train_sampling_strategy="group_by_length" if cfg.group_by_length else "random",
        dataloader_num_workers=2, remove_unused_columns=False, report_to="none", seed=cfg.seed,
    )
    trainer = AnswerOnlyTrainer(model=model, args=training_args, train_dataset=train_ds,
                                eval_dataset=eval_ds or None, data_collator=collator, processing_class=tokenizer)
    trainer.create_optimizer()
    steps = -(-len(train_ds) // (cfg.batch_size * cfg.grad_accum)) * cfg.epochs
    print(f"{steps:.0f} optimizer steps (effective batch {cfg.batch_size * cfg.grad_accum}, lr {cfg.lr}, {precision})")

    # the answer-only loss must match the model's own loss
    batch = {k: v.to("cuda") for k, v in collator([train_ds[i] for i in range(cfg.batch_size)]).items()}
    model.eval()
    with torch.no_grad(), trainer.accelerator.autocast():
        ours, ref = trainer.compute_loss(model, batch).item(), model(**batch).loss.item()
    print(f"loss check: answer-only {ours:.4f} vs default {ref:.4f}")
    assert abs(ours - ref) < 1e-2

    # memory check: forward + backward on the longest samples (the trainer starts with those) + the AdamW state
    longest = sorted(range(len(train_ds)), key=lambda i: len(train_ds.items[i]["input_ids"]))[-cfg.batch_size:]
    big = {k: v.to("cuda") for k, v in collator([train_ds[i] for i in longest]).items()}
    model.train()
    torch.cuda.reset_peak_memory_stats()
    trainer.compute_loss(model, big).backward()
    model.zero_grad(set_to_none=True)
    adam_gb = 2 * 4 * sum(p.numel() for p in model.parameters() if p.requires_grad) / 2**30
    need = torch.cuda.max_memory_allocated() / 2**30 + adam_gb
    have = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"memory: ~{need:.1f} GB needed (incl. {adam_gb:.1f} GB optimizer state), GPU has {have:.1f} GB")
    del big, batch
    torch.cuda.empty_cache()
    assert need < 0.85 * have, "won't fit: halve cfg.batch_size and double cfg.grad_accum"

    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    trainer.train(resume_from_checkpoint=True if any(cfg.output_dir.glob("checkpoint-*")) else None)
else:
    print("skipped: the model is already trained")

## 3. Check and save the model

In [ ]:
if RUN_TRAINING:
    print(trainer.evaluate())
    log = pd.DataFrame(trainer.state.log_history)
    display(log.dropna(subset=["loss"])[["step", "loss"]].tail())
    picks = [next(s for s in train_samples if s["task"] == t) for t in dict.fromkeys(s["task"] for s in train_samples)]
    show_generations(model, tokenizer, picks + [eval_samples["next_movie"][0]], "cuda")
    display(fact_recall(model, tokenizer, train_samples, "cuda").to_frame("trained on").style.format("{:.0%}"))

    final = model.to(torch.bfloat16)                 # Qwen's native dtype; ~1 GB
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    final.save_pretrained(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)
    print(f"saved to {FINAL_DIR}")
    del model, final, trainer                        # free the GPU for evaluation
    torch.cuda.empty_cache()

## 4. Evaluate on the 1,000 test users

Each user's **last** rating was never used for training or validation. The model writes titles with beam search
that can only spell real catalogue titles; movies the user already rated are skipped. Full ranking, the same users
and metrics as `sasrec.ipynb` and `tfrs.ipynb`.

In [ ]:
from tqdm.auto import tqdm

from genrec.data import Dataset, history, sample_test_users, target
from genrec.metrics import evaluate_lists
from genrec.models import qwen

ds = Dataset.load(DATA_DIR)
test_users = sample_test_users(ds.users)
rec = qwen.QwenRecommender(FINAL_DIR, ds.movies, device="cuda")

t0 = time.time()
metrics, tops = qwen.evaluate(rec, ds.sequences, test_users, "test", num_beams=30, batch_size=8,
                              progress=lambda it: tqdm(it, desc="beam search"))

popular = pd.Series([m for u in ds.users for m in history(ds.sequences[u], "test")]).value_counts().index.tolist()
pop_tops = [[m for m in popular if m not in set(history(ds.sequences[u], "test"))][:10] for u in test_users]
pop = evaluate_lists(pop_tops, [target(ds.sequences[u], "test") for u in test_users])
print(f"{len(test_users)} test users in {time.time() - t0:.0f}s")
results = pd.DataFrame({"Qwen2.5-0.5B": metrics, "most popular": pop}).T
results.to_csv(DRIVE_DIR / "qwen_ml32m_recent_test.csv")
results.style.format("{:.3f}")

## 5. Probabilities and the CSV for the API

For each user's top 10: P(the model answers exactly this title | their last 20 movies). Written to Drive and
downloaded to your computer - put it in `genrec/artifacts/lookup_inputs/` on your Mac.

In [ ]:
import math

rows = []
for u in tqdm(test_users, desc="scoring"):
    hist = history(ds.sequences[u], "test")
    titles = [rec.display[m] for m in tops[u]]
    for rank, (m, lp) in enumerate(zip(tops[u], rec.title_logprobs(hist, titles)), 1):
        rows.append({"user_id": u, "rank": rank, "movie_id": m, "probability": math.exp(lp), "log_probability": lp})
out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print(f"saved {len(out)} rows for {out.user_id.nunique()} users -> {OUT_CSV}")

from google.colab import files

files.download(str(OUT_CSV))     # your browser asks where to save it
out.groupby("rank")["probability"].describe()[["mean", "50%", "max"]].round(4)

## 6. Some examples

In [ ]:
titles = ds.titles()
for u in pd.Series(test_users).sample(5, random_state=1):
    tgt = titles[target(ds.sequences[u], "test")]
    top = [titles[m] for m in tops[u]]
    print(f"user {u}  last rated: {'; '.join(titles[m] for m in history(ds.sequences[u], 'test')[-5:])}")
    print(f"   real next: {tgt}   {f'HIT at rank {top.index(tgt) + 1}' if tgt in top else 'miss'}")
    print(f"   top 10:    {'; '.join(top)}\n")